In [20]:
import numpy as np
import jax
import jax.numpy as jnp
from safetensors import safe_open
from safetensors.numpy import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download

In [21]:
from abc import ABC, abstractmethod


class Model(ABC):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    @abstractmethod
    def next_token(self, input_text: str) -> str:
        pass

    def generate(self, input_text: str, max_tokens: int = 100) -> str:
        text = input_text
        eos_token = getattr(self.tokenizer, "eos_token", None)

        for _ in range(max_tokens):
            next_token = self.next_token(text)
            text += next_token
            if eos_token and next_token == eos_token:
                break

        return text


In [22]:
def softmax(x, axis=-1):
    return jax.nn.softmax(x, axis=axis)

In [23]:
class GELU:
    def __call__(self, x):
        return jax.nn.gelu(x, approximate=True)

    def __str__(self):
        return "GELU"

In [24]:
class Linear:
    def __init__(self, weights, bias) -> None:
        self.weights = weights
        self.bias = bias

    def __call__(self, x):
        return x @ self.weights + self.bias

    def __str__(self):
        return f"Linear weights: {self.weights.shape}, bias: {self.bias.shape}"

In [25]:
class LayerNorm:
    def __init__(self, weight, bias, eps=1e-5):
        self.weight = weight
        self.bias = bias
        self.eps = eps

    def __call__(self, x):
        mean = jnp.mean(x, axis=-1, keepdims=True)
        var = jnp.var(x, axis=-1, keepdims=True)

        x_norm = (x - mean) / jnp.sqrt(var + self.eps)

        return self.weight * x_norm + self.bias

    def __str__(self):
        return f"LayerNorm weights: {self.weight.shape}, bias: {self.bias.shape}"

In [26]:
class Attention:
    def __init__(self, attn_weights, attn_bias, proj_weights, proj_bias) -> None:
        self.attn_weights = attn_weights
        self.attn_bias = attn_bias
        self.proj_weights = proj_weights
        self.proj_bias = proj_bias

    def __call__(self, x):
        qkv = x @ self.attn_weights + self.attn_bias
        q, k, v = np.split(qkv, 3, axis=-1)
        scores = q @ k.T
        scores = scores / np.sqrt(k.shape[-1])
        scores = softmax(scores)
        out = scores @ v
        out = out @ self.proj_weights + self.proj_bias
        return out

    def __str__(self):
        return f"Attention weights: {self.attn_weights.shape}, bias: {self.attn_bias.shape}, proj weights: {self.proj_weights.shape}, proj bias: {self.proj_bias.shape}"

In [27]:
class MultiHeadAttention:
    def __init__(
        self, attn_weights, attn_bias, proj_weights, proj_bias, num_heads=8
    ) -> None:
        self.attn_weights = attn_weights
        self.attn_bias = attn_bias
        self.proj_weights = proj_weights
        self.proj_bias = proj_bias
        self.num_heads = num_heads

    def __call__(self, x):
        seq_len = x.shape[0]
        hidden_dim = x.shape[1]  # Should be 512
        head_dim = hidden_dim // self.num_heads  # 512 // 8 = 64

        # 1. Project input to Q, K, V
        qkv = x @ self.attn_weights + self.attn_bias  # (seq_len, 512*3)
        q, k, v = jnp.split(qkv, 3, axis=-1)  # Each: (seq_len, 512)

        # 2. Split into multiple heads
        # Reshape from (seq_len, 512) to (seq_len, num_heads, head_dim)
        q = q.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        k = k.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        v = v.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)

        # 3. Transpose to (num_heads, seq_len, head_dim) for batch processing
        q = q.transpose(1, 0, 2)  # (8, seq_len, 64)
        k = k.transpose(1, 0, 2)  # (8, seq_len, 64)
        v = v.transpose(1, 0, 2)  # (8, seq_len, 64)

        # 4. Compute attention scores for all heads
        scores = q @ k.transpose(0, 2, 1)  # (8, seq_len, seq_len)
        scores = scores / jnp.sqrt(head_dim)  # Scale by sqrt(64)

        # 5. Apply causal mask (prevent attending to future tokens)
        causal_mask = jnp.tril(jnp.ones((seq_len, seq_len)))  # Lower triangular matrix
        scores = jnp.where(causal_mask == 0, -1e10, scores)  # Mask future positions

        # 6. Apply softmax to get attention weights
        attn_weights = softmax(scores, axis=-1)  # (8, seq_len, seq_len)

        # 7. Apply attention weights to values
        output = attn_weights @ v  # (8, seq_len, 64)

        # 8. Concatenate heads back together
        output = output.transpose(1, 0, 2)  # (seq_len, 8, 64)
        output = output.reshape(seq_len, hidden_dim)  # (seq_len, 512)

        # 9. Final projection
        output = output @ self.proj_weights + self.proj_bias

        return output

    def __str__(self):
        return f"MultiHeadAttention weights: {self.attn_weights.shape}, bias: {self.attn_bias.shape}, proj weights: {self.proj_weights.shape}, proj bias: {self.proj_bias.shape}"

In [28]:
class GPT2(Model):
    def __init__(self) -> None:
        tokenizer = AutoTokenizer.from_pretrained(
            "erwanf/gpt2-mini", cache_dir="./data/hf"
        )
        super().__init__(tokenizer)

        model_file = hf_hub_download(
            repo_id="erwanf/gpt2-mini",
            filename="model.safetensors",
            cache_dir="./data/hf",
        )

        self.model_weights = load_file(model_file)

        self.attention_blocks = []
        self.ffn_blocks = []

        for i in range(4):
            # Attention block (LayerNorm + Attention)
            ln_1_weight = self.model_weights[f"transformer.h.{i}.ln_1.weight"]
            ln_1_bias = self.model_weights[f"transformer.h.{i}.ln_1.bias"]
            attn_c_attn_weight = self.model_weights[f"transformer.h.{i}.attn.c_attn.weight"]
            attn_c_attn_bias = self.model_weights[f"transformer.h.{i}.attn.c_attn.bias"]
            attn_c_proj_weight = self.model_weights[f"transformer.h.{i}.attn.c_proj.weight"]
            attn_c_proj_bias = self.model_weights[f"transformer.h.{i}.attn.c_proj.bias"]

            self.attention_blocks.append(
                {
                    "ln": LayerNorm(ln_1_weight, ln_1_bias),
                    "attn": MultiHeadAttention(
                        attn_c_attn_weight,
                        attn_c_attn_bias,
                        attn_c_proj_weight,
                        attn_c_proj_bias,
                    ),
                }
            )

            # FFN block (LayerNorm + Linear + GELU + Linear)
            ln_2_weight = self.model_weights[f"transformer.h.{i}.ln_2.weight"]
            ln_2_bias = self.model_weights[f"transformer.h.{i}.ln_2.bias"]
            ffn_1_weight = self.model_weights[f"transformer.h.{i}.mlp.c_fc.weight"]
            ffn_1_bias = self.model_weights[f"transformer.h.{i}.mlp.c_fc.bias"]
            ffn_2_weight = self.model_weights[f"transformer.h.{i}.mlp.c_proj.weight"]
            ffn_2_bias = self.model_weights[f"transformer.h.{i}.mlp.c_proj.bias"]

            self.ffn_blocks.append(
                {
                    "ln": LayerNorm(ln_2_weight, ln_2_bias),
                    "linear1": Linear(ffn_1_weight, ffn_1_bias),
                    "gelu": GELU(),
                    "linear2": Linear(ffn_2_weight, ffn_2_bias),
                }
            )

    def next_token(self, input_text):
        input_tokens = self.tokenizer.encode(input_text)
        token_embeddings = self.model_weights["transformer.wte.weight"][input_tokens]
        pos_embeddings = self.model_weights["transformer.wpe.weight"][
            : len(input_tokens)
        ]
        x = token_embeddings + pos_embeddings

        for attn_block, ffn_block in zip(self.attention_blocks, self.ffn_blocks):
            # Attention with residual
            attn_out = attn_block["attn"](attn_block["ln"](x))
            x += attn_out  # ← Residual connection

            # FFN with residual
            ffn_out = ffn_block["linear2"](
                ffn_block["gelu"](ffn_block["linear1"](ffn_block["ln"](x)))
            )
            x += ffn_out  # ← Residual connection

        # final layer norm
        x = LayerNorm(
            self.model_weights["transformer.ln_f.weight"], self.model_weights["transformer.ln_f.bias"]
        )(x)

        logits = x @ self.model_weights["transformer.wte.weight"].T
        token_ids = np.argmax(logits, axis=-1)

        next_token_id = int(token_ids[-1])
        return self.tokenizer.decode(next_token_id)


In [29]:
text = "Elon Musk is"

gpt2 = GPT2(
result = gpt2.generate(text, max_tokens=10)
print(result)

Elon Musk is a former NASA astronaut and a former NASA astronaut.


In [30]:
class RMSNorm():
    def __init__(self, w, eps=1e-5) -> None:
        self.w = w
        self.eps = eps

    def __call__(self, x):
        rms = jnp.sqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + self.eps)
        return (x / rms) * self.w

In [31]:
class SmolLM2MLP():
    def __init__(self, gate_proj, up_proj, down_proj):
        self.gate_proj = gate_proj
        self.up_proj = up_proj
        self.down_proj = down_proj

    def swiglu(self, x):
        # Gate path
        gate = jnp.dot(x, self.gate_proj.T)  # Project to intermediate dim
        gate = jax.nn.silu(gate)         # Apply SiLU activation

        # Value path
        value = jnp.dot(x, self.up_proj.T)    # Project to intermediate dim

        # Combine
        hidden = gate * value             # Element-wise multiply

        # Output
        output = jnp.dot(hidden, self.down_proj.T)  # Project back to d_model

        return output

    def __call__(self, x):
        return self.swiglu(x)


In [32]:
class SmolLM2TransformerBlock():
    def __init__(self, weights) -> None:
        self.weights = weights

    def __call__(self, x):
        ans = x.copy()

        layer_1 = [
            RMSNorm(self.weights["input_layernorm"]),
            GroupedQueryAttention(
                self.weights["self_attn_q_proj"],
                self.weights["self_attn_k_proj"],
                self.weights["self_attn_v_proj"],
                self.weights["self_attn_o_proj"],
            ),
        ]
        layer_2 = [
            RMSNorm(self.weights["post_attention_layernorm"]),
            SmolLM2MLP(
                self.weights["mlp_gate_proj"],
                self.weights["mlp_up_proj"],
                self.weights["mlp_down_proj"],
            )
        ]

        layer_result = ans
        for layer in layer_1:
            layer_result = layer(layer_result)
        ans += layer_result

        layer_result = ans.copy()
        for layer in layer_2:
            layer_result = layer(layer_result)
        ans += layer_result

        return ans


In [33]:
class SmolLM2(Model):
    def __init__(self) -> None:
        self.layers = []

        tokenizer = AutoTokenizer.from_pretrained(
            "HuggingFaceTB/SmolLM2-135M-Instruct",
            cache_dir="./data/hf",
        )
        super().__init__(tokenizer)

        model_file = hf_hub_download(
            repo_id="HuggingFaceTB/SmolLM2-135M-Instruct",
            filename="model.safetensors",
            cache_dir="./data/hf",
        )

        self.model_weights = {}
        with safe_open(model_file, framework="jax") as f:
            for key in f.keys():
                tensor = f.get_tensor(key)
                if hasattr(tensor, "dtype") and str(tensor.dtype) == "bfloat16":
                    tensor = tensor.astype(np.float32)
                self.model_weights[key] = tensor

        layers_count = 30
        for i in range(layers_count):
            weights = {
                "input_layernorm": self.model_weights[f"model.layers.{i}.input_layernorm.weight"],
                "mlp_down_proj": self.model_weights[f"model.layers.{i}.mlp.down_proj.weight"],
                "mlp_gate_proj": self.model_weights[f"model.layers.{i}.mlp.gate_proj.weight"],
                "mlp_up_proj": self.model_weights[f"model.layers.{i}.mlp.up_proj.weight"],
                "post_attention_layernorm": self.model_weights[f"model.layers.{i}.post_attention_layernorm.weight"],
                "self_attn_k_proj": self.model_weights[f"model.layers.{i}.self_attn.k_proj.weight"],
                "self_attn_o_proj": self.model_weights[f"model.layers.{i}.self_attn.o_proj.weight"],
                "self_attn_q_proj": self.model_weights[f"model.layers.{i}.self_attn.q_proj.weight"],
                "self_attn_v_proj": self.model_weights[f"model.layers.{i}.self_attn.v_proj.weight"],
            }
            self.layers.append(SmolLM2TransformerBlock(weights))

    def next_token(self, input_text: str) -> str:
        input_tokens = jnp.array(self.tokenizer.encode(input_text))
        token_embeddings = self.model_weights["model.embed_tokens.weight"][input_tokens]

        x = token_embeddings
        for layer in self.layers:
            x = layer(x)

        x = RMSNorm(self.model_weights["model.norm.weight"])(x)

        logits = x @ self.model_weights["model.embed_tokens.weight"].T
        token_ids = jnp.argmax(logits[-1, :], axis=-1)

        next_token_id = int(token_ids)
        return self.tokenizer.decode([next_token_id])


In [34]:
def rope(x, dim, base=10000.0):
    seq_len = x.shape[-2]

    # Compute frequencies
    inv_freq = 1.0 / (base ** (jnp.arange(0, dim, 2).astype(jnp.float32) / dim))

    # Compute angles for each position
    t = jnp.arange(seq_len, dtype=jnp.float32)
    freqs = jnp.outer(t, inv_freq)  # (seq_len, dim/2)

    # Duplicate frequencies for both halves
    emb = jnp.concatenate([freqs, freqs], axis=-1)  # (seq_len, dim)

    cos = jnp.cos(emb)
    sin = jnp.sin(emb)

    # Rotate half
    x1 = x[..., :dim//2]
    x2 = x[..., dim//2:]
    x_rotated = jnp.concatenate([-x2, x1], axis=-1)

    # Apply rotation
    return x * cos + x_rotated * sin

class GroupedQueryAttention():
    def __init__(self, wq, wk, wv, wo, q_count=9, kv_count=3) -> None:
        self.wq, self.wk, self.wv, self.wo = wq, wk, wv, wo
        self.q_count = q_count
        self.kv_count = kv_count
        self.head_dim = 576 // q_count

    def __call__(self, x):
        seq_len = x.shape[0]

        # project
        q = x @ self.wq.T
        k = x @ self.wk.T
        v = x @ self.wv.T

        # reshape from (seq_len, heads, dim) -> (heads, seq_len, dim)
        q = q.reshape(seq_len, self.q_count, self.head_dim).transpose(1, 0, 2)
        k = k.reshape(seq_len, self.kv_count, self.head_dim).transpose(1, 0, 2)
        v = v.reshape(seq_len, self.kv_count, self.head_dim).transpose(1, 0, 2)

        # rope
        q, k = rope(q, self.head_dim), rope(k, self.head_dim)

        # expand k and v to match q
        k = k.repeat(3, axis=0)
        v = v.repeat(3, axis=0)

        # attention
        attn_scores = (q @ k.transpose(0, 2, 1)) / jnp.sqrt(self.head_dim)
        mask = jnp.tril(jnp.ones((seq_len, seq_len)))
        attn_scores = jnp.where(mask == 0, -1e9, attn_scores)
        attn_probs = jax.nn.softmax(attn_scores, axis=-1)
        output = attn_probs @ v

        # merge heads and project (heads, token, dim) -> (token, heads, dim) -> concatenated (token, dim)
        output = output.transpose(1, 0, 2).reshape(seq_len, 576)

        return output @ self.wo.T


In [35]:
smollm2 = SmolLM2()
messages = [
    {"role": "user", "content": "What is the meaning of life?"},
]
prompt = smollm2.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
smollm2.generate(prompt, max_tokens=10)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is the meaning of life?<|im_end|>\n<|im_start|>assistant\nThe meaning of life is a topic of philosophical inquiry'

In [36]:
for i in smollm2.model_weights.keys():
    print(i, smollm2.model_weights[i].shape)

model.embed_tokens.weight (49152, 576)
model.layers.0.input_layernorm.weight (576,)
model.layers.0.mlp.down_proj.weight (576, 1536)
model.layers.0.mlp.gate_proj.weight (1536, 576)
model.layers.0.mlp.up_proj.weight (1536, 576)
model.layers.0.post_attention_layernorm.weight (576,)
model.layers.0.self_attn.k_proj.weight (192, 576)
model.layers.0.self_attn.o_proj.weight (576, 576)
model.layers.0.self_attn.q_proj.weight (576, 576)
model.layers.0.self_attn.v_proj.weight (192, 576)
model.layers.1.input_layernorm.weight (576,)
model.layers.1.mlp.down_proj.weight (576, 1536)
model.layers.1.mlp.gate_proj.weight (1536, 576)
model.layers.1.mlp.up_proj.weight (1536, 576)
model.layers.1.post_attention_layernorm.weight (576,)
model.layers.1.self_attn.k_proj.weight (192, 576)
model.layers.1.self_attn.o_proj.weight (576, 576)
model.layers.1.self_attn.q_proj.weight (576, 576)
model.layers.1.self_attn.v_proj.weight (192, 576)
model.layers.10.input_layernorm.weight (576,)
model.layers.10.mlp.down_proj.wei